In [4]:
import json
from pathlib import Path
from urllib.request import Request, urlopen
from bs4 import BeautifulSoup

def get_html(url):
    request = Request(url, headers={"User-Agent": "Mozilla/5.0 (Python exercise)"})
    with urlopen(request, timeout=30) as response:
        return response.read().decode("utf-8")

output_dir = Path.cwd()
if output_dir.name != "day_22":
    output_dir = output_dir / "day_22"
output_dir.mkdir(parents=True, exist_ok=True)

bu_url = "https://www.bu.edu/president/boston-university-facts-stats/"
bu_soup = BeautifulSoup(get_html(bu_url), "html.parser")
bu_facts = {"Quick Facts": {}, "Community": [], "Campus": {}, "Academics": []}

quick_labels = {"Research Expenditures", "Study Abroad Programs", "Sponsored Research Awards"}
campus_labels = {"Classrooms", "Buildings", "Laboratories", "Libraries", "Campus Area (acres)"}

for heading in bu_soup.find_all("h3"):
    label = heading.get_text(" ", strip=True)
    if label in quick_labels or label in campus_labels:
        sibling = heading.find_next_sibling("div")
        value = sibling.get_text(" ", strip=True) if sibling else ""
        if label in quick_labels:
            bu_facts["Quick Facts"][label] = value
        else:
            bu_facts["Campus"][label] = value

for category_heading in bu_soup.find_all("h4"):
    category = category_heading.get_text(" ", strip=True)
    if category not in {"Community", "Academics"}:
        continue
    items = []
    for sibling in category_heading.find_all_next(["h4", "li"]):
        if sibling is not category_heading and sibling.name == "h4":
            break
        if sibling.name == "li":
            item = sibling.get_text(" ", strip=True)
            if item:
                items.append(item)
    bu_facts[category] = items

bu_output = output_dir / "bu_facts_stats.json"
bu_output.write_text(json.dumps(bu_facts, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved BU facts to {bu_output}")

Saved BU facts to c:\Users\user\Projects\30-Days-of-pythonexercise\day_22\bu_facts_stats.json


In [5]:
uci_url = "https://archive.ics.uci.edu/datasets"
uci_soup = BeautifulSoup(get_html(uci_url), "html.parser")
datasets = []
seen_urls = set()

for heading in uci_soup.find_all("h2"):
    name = heading.get_text(" ", strip=True)
    link = heading.find_previous("a", href=True)
    if not name or not link:
        continue
    href = link.get("href", "")
    dataset_url = href if href.startswith("http") else "https://archive.ics.uci.edu" + href
    if dataset_url in seen_urls:
        continue
    seen_urls.add(dataset_url)
    card = heading.parent
    description = ""
    for paragraph in card.find_all(["p", "div"], recursive=True):
        text = paragraph.get_text(" ", strip=True)
        if text and text != name:
            description = text
            break
    datasets.append({"name": name, "description": description, "url": dataset_url})

uci_output = output_dir / "uci_datasets.json"
uci_output.write_text(json.dumps(datasets, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved {len(datasets)} UCI datasets to {uci_output}")

Saved 10 UCI datasets to c:\Users\user\Projects\30-Days-of-pythonexercise\day_22\uci_datasets.json


In [6]:
presidents_url = "https://en.wikipedia.org/wiki/List_of_presidents_of_the_United_States"
president_soup = BeautifulSoup(get_html(presidents_url), "html.parser")
presidents_table = None

for table in president_soup.find_all("table"):
    headers = [header.get_text(" ", strip=True).lower() for header in table.find_all("th")]
    if any("president" in header or "name" in header for header in headers):
        presidents_table = table
        break

presidents = []
if presidents_table:
    for row in presidents_table.find_all("tr"):
        values = [cell.get_text(" ", strip=True) for cell in row.find_all(["th", "td"])]
        if not values or not values[0].isdigit():
            continue
        name = next((value for value in values if "(" in value and any(year in value for year in ("17", "18", "19", "20"))), "")
        term = next((value for value in values if "?" in value and any(char.isdigit() for char in value) and "(" not in value), "")
        party = next((value for value in values if value in {"Unaffiliated", "Federalist", "Democratic-Republican", "Democratic- Republican", "Democratic", "Republican", "Whig", "National Union"}), "")
        if name:
            presidents.append({"number": values[0], "name": name, "term": term, "party": party})

presidents_output = output_dir / "us_presidents.json"
presidents_output.write_text(json.dumps(presidents, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved {len(presidents)} presidents to {presidents_output}")

Saved 47 presidents to c:\Users\user\Projects\30-Days-of-pythonexercise\day_22\us_presidents.json
